# Classification with a generative decoder

This notebook belongs to the [DGPs 2026 workshop *Text Classification with Open Source Models*](https://llm-content-analysis.com/sections/workshop/index.html).

You'll classify the same open-ended survey text from the [Real World Worry Waves Dataset (RW3D)](https://osf.io/9b85r/) as in the previous notebook — this time with a small, open-source **generative decoder** model.

Instead of a fixed set of labels, you write a prompt, the model generates free text, and you parse that text back into a structured answer.

**How to use this notebook:** run each cell from top to bottom with `Shift+Enter` or by clicking the run button. Most cells are given and fully explained; a few are marked **Your turn** — those are where you edit or write something yourself. Every exercise cell has a safe default, so nothing breaks if you leave it as-is, but you'll get more out of the session if you actually change it.

**Important note for Colab users:** If you are running this notebook in Google Colab, please follow these steps: 1. Make sure you have a Google account and are logged in. 2. To create a copy of this notebook in your own Google Drive (advised), select "Copy to Drive" at the top of the Colab page. 3. Once the notebook copy opens in Colab, click on "Runtime" in the menu bar, then select "Change runtime type" and choose "T4 GPU" as the hardware accelerator. 4. Click "Save" to apply the changes. 5. Now you can run the notebook cells as instructed.

## Setup

First, we install the Python packages we need. This will take some time to complete.

In [ ]:
%pip install -q transformers accelerate pandas

Next, we can change the logging level of the `transformers` library to avoid cluttering the notebook with warnings and info messages.

You do not have to run this cell, but it will make the output easier to read.

In [1]:
from transformers import logging
logging.set_verbosity_error()

## Code-Along

We'll build the classification workflow together.

There are a couple of small **"your turn"** fill-ins along the way that you can work on after the code-along.

## Step 1: Load the data

We'll use the same curated sample of survey responses from RW3D as in the previous notebook.

First, let's load the data. Because generative decoder models are slower than encoder models, we'll only use a small sample of 10 texts for this notebook. You can change the number of rows to load by changing the argument to `head()` in the next cell.

In [2]:
#| label: load-data
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/felixdidi/llm-content-analysis/main/sections/workshop/data/rw3d_workshop_sample.csv"
df = pd.read_csv(DATA_URL).head(10)
df

,id,text,main_emotion,anger,fear,sadness
0,1,Stressed and uninformed. Don't feel enough is ...,anger,9,1,6
1,2,"I feel worried about the virus, burnout about ...",fear,8,9,7
2,3,Disgusted how unprotected NHS staff are and ho...,sadness,2,6,7
3,4,I am not too worried about the situation as I ...,sadness,1,3,6
4,5,I'm finding the Corona situation quite dauntin...,sadness,5,2,7
5,6,At the moment I feel like I am doing everythin...,sadness,1,2,2
6,7,I feel sad for all the people infected and tho...,sadness,2,3,4
7,8,I'm worried about elderly parents. I'm worried...,anger,9,7,8
8,9,I am very anxious and worried about the COVID-...,fear,2,9,9
9,10,i'm very Angry at the moment as there are many...,anger,8,8,6


## Step 2: Load a small open-source decoder model

We'll use [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), an instruction-tuned model small enough to run comfortably on Colab's free GPU (and likely also your own laptop, if you intend to run it locally).

Unlike the encoder models from the previous notebook, this one wasn't trained to return scores for a fixed set of labels. Instead, it generates a stream of text, so we'll need to tell it what we want in the prompt itself, and turn its answer back into something structured ourselves.

Generative decoder models are also much larger than encoder models. Loading the model may take some time.

In [3]:
from transformers import pipeline

generate = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
)

We write a short system prompt that states the task and asks for a strict JSON answer — here, whether the text expresses `fear` (`true`/`false`) plus a short `reasoning`. Because the prompt doesn't show the model any examples, classifications will be based only on the context knowledge of the model. This is called **zero-shot** prompting.

Unlike the encoder pipelines from the previous notebook, a decoder's `generate()` call doesn't batch cleanly over a list of texts out of the box — so we call it once per text, in a `for` loop. `do_sample=False` keeps answers deterministic, so re-running the cell gives the same result. Running it on all texts may take some time.

In [4]:
#| label: classify
import json

results = []
for text in df['text']:
    instructions = [
        {"role": "system",
         "content": "You are a trained assistant for content analysis who determines whether a text expresses fear. Always answer precisely in JSON format with fear (true or false) and reasoning (a short justification in English). Return only the JSON response, starting with '{' and ending with '}', with these two parameters."},
        {"role": "user", "content": text}]
    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    results.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

results

[{'text': "Stressed and uninformed. Don't feel enough is being done by the government to help with the NHS crisis or the proper enforcement of social distancing.nhs staff should be tested immediately and given the right equipment to fight the Corona virus. There should be a full lockdown straight away to stop the spread.the equipment side of it is the fact that they aren't really getting enough or even any at all.how can you be expected to take urgent care of someone, and also be worried that you might catch it,or maybe even already have it.",
  'response': '```json\n{"fear": true, "reasoning": "The text expresses significant concern about the current situation regarding the NHS crisis, lack of action from the government, inadequate testing and equipment for healthcare workers, and the need for immediate measures such as a full lockdown to control the spread of the coronavirus. This indicates a high level of anxiety and fear related to public health issues."}\n```'},
 {'text': 'I feel 

**Basically, that was it.** We now have a response for every text in the sample. `results` is just a list of dictionaries — one per text, each holding the original `text` and the model's raw `response` string — and that's exactly the structure of a JSON file. So, you could simply save it with Python's built-in `json` module:

```python
with open("results.json", "w") as f:
    json.dump(results, f)
```

That file could be picked up in R or any other tool from here. We'll keep working in Python below.

If we want to keep working in Python, we need to parse the model's free-text output back into a structured answer. We use some simple string manipulation to extract the `fear` value and the `reasoning` text. This is a bit fragile because the model's output may not always be perfectly formatted. For example, we can see that, although we told the model to return a strict JSON answer beginning with `{`, the model consistently returns the JSON object wrapped in a markdown code block (` ```json\n{...}\n``` `). We can strip that away with some simple string manipulation, which is already implemented in the `parse_response()` function below. 

In [5]:
#| label: parse
def parse_response(response):
    response = response.strip().strip("`").removeprefix("json").strip()
    try:
        parsed = json.loads(response)
        return parsed['fear'], parsed['reasoning']
    except (json.JSONDecodeError, KeyError):
        return None, None

parsed_data = [(entry['text'], *parse_response(entry['response'])) for entry in results]
parsed_data = pd.DataFrame(parsed_data, columns=['text', 'fear', 'reasoning'])
parsed_data

,text,fear,reasoning
0,Stressed and uninformed. Don't feel enough is ...,True,The text expresses significant concern about t...
1,"I feel worried about the virus, burnout about ...",True,The text expresses significant worry about mul...
2,Disgusted how unprotected NHS staff are and ho...,True,The text expresses strong negative emotions su...
3,I am not too worried about the situation as I ...,False,The text does not express any significant leve...
4,I'm finding the Corona situation quite dauntin...,False,"The text expresses hope, resilience, and solid..."
5,At the moment I feel like I am doing everythin...,False,The text does not express any significant leve...
6,I feel sad for all the people infected and tho...,True,The text expresses concern about the pandemic'...
7,I'm worried about elderly parents. I'm worried...,True,The text expresses multiple concerns including...
8,I am very anxious and worried about the COVID-...,True,The text expresses significant anxiety and wor...
9,i'm very Angry at the moment as there are many...,True,The text expresses concern and anxiety about t...


### Your turn

The prompt above only asks about `fear`. Edit the system prompt so it asks for `anger`, `fear`, and `sadness` all at once (three booleans plus one `reasoning`, in a single JSON object). If everything works correctly, the code below should pull out all three fields instead of just `fear`. Re-run the cell — how many responses still parse cleanly?

**Feeling bold?** If parsing below fails for some responses, you can try to improve the parsing function to handle more edge cases. Look at the raw output to see what the model is returning, and try to make the parsing more robust. You can also try to improve the prompt itself to get cleaner output from the model or increase the maximum number of tokens in the `generate()` call if the model is truncating its output.

In [ ]:
results_all_three = []
for text in df['text']:
    instructions = [
        {"role": "system",
         "content": "ADD YOUR UPDATED SYSTEM PROMPT HERE"}, # TODO
        {"role": "user", "content": text}]
    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    results_all_three.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

results_all_three

In [ ]:
def parse_response_all_three(response):
    response = response.strip().strip("`").removeprefix("json").strip()
    try:
        parsed = json.loads(response)
        return parsed['fear'], parsed['anger'], parsed['sadness'], parsed['reasoning']
    except (json.JSONDecodeError, KeyError):
        return None, None, None, None

parsed_data_all_three = [(entry['text'], *parse_response_all_three(entry['response'])) for entry in results_all_three]
parsed_data_all_three = pd.DataFrame(parsed_data_all_three, columns=['text', 'fear', 'anger', 'sadness', 'reasoning'])
parsed_data_all_three

## Exercise: From zero-shot to few-shot prompting

So far, the model has only ever seen the text it needs to classify — no examples of what a "correct" answer looks like. That's **zero-shot** prompting. In **few-shot** prompting, you show the model one or more examples (e.g., an example text, maybe the JSON answer you'd want for it) before asking about the real text. This can nudge the model toward your specific understanding of the category and toward the exact output format you asked for. However, it may also bias the model toward the examples you show it, so you have to be careful about what examples you choose.

Below, write 1–2 short example texts, together with the JSON answer you think is correct for each — base this on your own understanding of what should count as `fear` here. These examples will be inserted into the prompt before the real text every time the model is called.

In [ ]:
# TODO: write 1-2 short examples of your own: a text, and the JSON answer
# you think is correct for it (based on your own understanding of "fear").
# These will be shown to the model as worked examples before it classifies
# the real texts below.
few_shot_examples = [
    {
        "text": "EXAMPLE TEXT 1",
        "response": 'EXAMPLE RESPONSE 1',
    },
    {
        "text": "EXAMPLE TEXT 2",
        "response": 'EXAMPLE RESPONSE 2',
    },
]

Now we run the same loop as above, but this time we add your examples as extra turns in the conversation before the real text — each example becomes one `user` turn (the example text) followed by one `assistant` turn (the example answer), exactly how it would look if the model had already answered it.

In [ ]:
results_fewshot = []
for text in df['text']:
    instructions = [
        {"role": "system",
         "content": "You are a trained assistant for content analysis who determines whether a text expresses fear. Always answer precisely in JSON format with fear (true or false) and reasoning (a short justification in English). Return only the JSON response, starting with '{' and ending with '}', with these two parameters."},
    ]
    for example in few_shot_examples:
        instructions.append({"role": "user", "content": example["text"]})
        instructions.append({"role": "assistant", "content": example["response"]})
    instructions.append({"role": "user", "content": text})

    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    results_fewshot.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

results_fewshot

Parse the few-shot responses the same way as before, and compare them to the zero-shot predictions from the code-along side by side.

In [ ]:
parsed_data_fewshot = [(entry['text'], *parse_response(entry['response'])) for entry in results_fewshot]
parsed_data_fewshot = pd.DataFrame(parsed_data_fewshot, columns=['text', 'fear', 'reasoning'])

comparison = parsed_data[['text', 'fear']].merge(
    parsed_data_fewshot[['text', 'fear']],
    on='text', suffixes=('_zero_shot', '_few_shot'),
)
comparison

Look at where the two columns disagree. Few-shot prompting can make a model's answers more consistent with *your* specific definition of a category, and can improve how reliably it sticks to the requested format — but it isn't free. The examples you write inevitably steer the model's judgment (a form of researcher bias that's easy to underestimate with just 1–2 examples), each call now sends a longer prompt through the model (this loop was already slow; more examples means more tokens, means more time), and there's no guarantee the model is actually applying your examples' *reasoning* to a new, genuinely ambiguous case rather than just imitating their tone or length.

### Bonus: does a smaller model still hold up?

If you have time left, try the (zero-shot) classification from the code-along again, but with an even smaller model this time: [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), a third of the size of the one we've used so far. Does it still reliably return valid JSON, or do more responses fail to parse than with the 1.5B model? Where it does parse, do the label decisions still look reasonable to you?

In [ ]:
alternative_generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

In [ ]:
# TODO: reuse the loop from Step 3 (swap generate for alternative_generator) to
# classify df['text'] again, then parse the responses with parse_response and
# check how well the smaller model performs compared to the larger one. 